In [2]:
import argparse
import time

import numpy as np

from scipy import integrate
from scipy.optimize import minimize_scalar, root_scalar

import matplotlib.pyplot as plt

import warnings
from tqdm import tqdm

# warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)



In [3]:



class OneSided():
  def __init__(self, args):
        self.b=args.b
        self.s=args.s
        self.r=args.r

        self.MIN_B=args.MIN_B
        self.MAX_B=args.MAX_B
        self.MIN_S=args.MIN_S
        self.MAX_S=args.MAX_S

        self.bh = args.bh
        self.bl = args.bl

        self.sh = args.sh
        self.sl = args.sl


        if args.c != None:
            self.c_b = args.c
            self.c_s = args.c
            self.c = args.c
            self.kappa = self.c/self.r
            self.b_diamond = self.s

        else:
            self.c_s=args.c_s
            self.c_b=args.c_b
            self.b_diamond = min(self.bh, self.s + np.abs(self.c_s-self.c_b)/self.r)





        self.b_star_c = self.b_star_closed()
        self.b_dag_c = self.b_dag_closed(b_condition=self.b_star_c)


  def p_R(self, b, s):
      if self.r*(b-s)>=np.abs(self.c_s-self.c_b):
          return (b+s)/2-(self.c_s-self.c_b)/(2*self.r)
      elif self.r*(b-s)<(self.c_b-self.c_s):
          return b
      else:
          return s

  def p_R_partial_b(self, b, s):
      if self.r*(b-s)>=np.abs(self.c_s-self.c_b):
          return 1/2
      elif self.r*(b-s)<(self.c_b-self.c_s):
          return 1
      else:
          return 0

  def delta(self, b, b_condition):
      def integrand_delta(z):
          numer = self.p_R_partial_b(b=z, s=self.s)
          denom = self.r*(z - self.p_R(b=z, s=self.s)) + self.c_b
          return numer/denom
      if (b<= b_condition) and (b>= max(self.s, self.bl)):
          result = integrate.quad(func=integrand_delta, a=b, b=b_condition)
          return result[0]
      else:
          return 0

  def exp_r_delta(self, b, b_condition):
      return np.exp(-self.r * self.delta(b=b, b_condition=b_condition))

  def exp_r_delta_c(self, b, b_condition):
      if b < b_condition:
          numer= (b-self.s)+(2*self.kappa)
          denom= (b_condition-self.s)+(2*self.kappa)
          return numer/denom
      else:
          return 1

  def expected_exp_r_delta(self):
      numer = self.b_dag_c - self.s + 2*self.kappa
      denom = self.b_star_c - self.s + 2*self.kappa
      result = (numer/denom) +1
      return result/2

  def delta_c(self, b, b_condition):
      return (np.log(self.exp_r_delta_c(b,b_condition)))/(-self.r)

  def v_tilde_b(self, b, b_condition):
      if (b<=b_condition) and (b>= max(self.s, self.bl)):
          result = self.exp_r_delta(b=b, b_condition=b_condition)*(b-self.p_R(b, self.s)+(self.c_b/self.r))-(self.c_b/self.r)
          return result
      elif (b> b_condition) and (b<= self.bh):
          return b-self.p_R(b_condition, self.s)
      else:
          return 0

  def v_tilde_b_closed(self, b, b_condition):
      if b >= max(self.s, self.bl):
          numer = (self.r*(b-self.s)+self.c_b+self.c_s)**2
          denom = 2*self.r*(self.r*(b_condition-self.s)+self.c_b+self.c_s)
          return (numer/denom) - (self.c_b/self.r)
      else:
          return 0

  def b_dag_opt(self, b_condition):
      if self.v_tilde_b(b=max(self.s, self.bl), b_condition=b_condition) >0:
          return max(self.s, self.bl)
      else:
          def b_dag_constraints(z):
              if self.v_tilde_b(b=z, b_condition=b_condition)<=0:
                  return -z
              else:
                  return 1000
          result = minimize_scalar(b_dag_constraints, bounds = (max(self.s, self.bl), self.bh), method='bounded')
          return result.x

  # Root로 구함
  def b_dag(self, b_condition):
      if self.v_tilde_b(b=max(self.s, self.bl), b_condition=b_condition) >0:
          return max(self.s, self.bl)
      else:
          # def b_dag_constraints(z):
          #     if self.v_tilde_b(b=z, b_condition=b_condition)<=0:
          #         return -z
          #     else:
          #         return 1000
          # result = minimize_scalar(b_dag_constraints, bounds = (max(self.s, self.bl), self.bh), method='bounded')
          # return result.x
          opt = root_scalar(self.v_tilde_b, bracket=[max(self.s, self.bl), self.bh], args=(b_condition))
          result = opt.root
          return result

  def b_dag_2(self, b_condition):
      if self.v_tilde_b(b=max(self.s, self.bl), b_condition=b_condition) >0:
          return max(self.s, self.bl)
      else:
          def b_dag_constraints(z):
              if self.v_tilde_b_closed(b=z, b_condition=b_condition)<=0:
                  return -z
              else:
                  return 1000
          result = minimize_scalar(b_dag_constraints, bounds = (max(self.s, self.bl), self.bh), method='bounded')
          return result.x

  def b_dag_closed(self, b_condition):
      def D(beta, b_condition):
          result = 2*(self.c_b+(self.r*beta))*(self.r*(b_condition-self.s)+self.c_s+self.c_b)
          return result
      def b_double_dag(beta, b_condition):
          if beta < self.v_tilde_b(b=max(self.bl, self.s), b_condition=b_condition):
              return max(self.bl, self.s)
          elif (beta >= self.v_tilde_b(b=max(self.bl, self.s), b_condition=b_condition)) and (beta<self.v_tilde_b_closed(b=b_condition, b_condition=b_condition)):
              numer = np.sqrt(D(beta=beta, b_condition=b_condition)) - self.c_s-self.c_b
              return self.s + (numer/self.r)
          elif (beta>=self.v_tilde_b_closed(b=b_condition, b_condition=b_condition)) and (beta<self.v_tilde_b_closed(b=self.bh, b_condition=b_condition)):
              return (b_condition+self.s)/2 + beta
      return b_double_dag(beta=0, b_condition=b_condition)

  def V_tilde_s(self, b, b_condition):
      if b >= b_condition:
          return self.p_R(b_condition, self.s)-self.s

      elif (b>= self.b_dag(b_condition=b_condition)) and (b<b_condition):
          numer = (b-self.s + (2*self.kappa))**2
          denom = 2*(b_condition-self.s+ (2*self.kappa))
          return (numer/denom)-self.kappa
      else:
          return 0

  def V_tilde_s_closed(self, b, b_condition):
      numer = (self.r*(b-self.s)+self.c_b+self.c_s)**2
      denom = 2*self.r*(self.r*(b_condition-self.s)+self.c_b+self.c_s)
      return max((numer/denom)-(self.c_s/self.r), 0)

  def V_tilde_b(self, b, b_condition):
      return max(0, self.v_tilde_b(b=b, b_condition=b_condition))

  def V_s_integrand(self, z, b_condition):
      return self.V_tilde_s(b=z, b_condition=b_condition)/(self.bh-self.bl)

  def V_s_obj(self, b_condition):
      result_1 = (self.p_R(b=b_condition, s=self.s)-self.s)*(self.bh- b_condition)/(self.bh-self.bl)
      b_dag = self.b_dag(b_condition=b_condition)
      result_2 = integrate.quad(func=self.V_s_integrand, a=b_dag, b=b_condition, args=(b_condition))[0]
      return -result_1 - result_2

  def b_star_V_s(self):
      result = minimize_scalar(fun=self.V_s_obj, bounds=(max(self.s, self.bl), self.bh), method='bounded')
      if result.success == True:
          return (result.x, -result.fun)
      else:
          raise Exception("b_star optimization failed")

  def V_S_closed(self):
      def V_s_integrand_closed(z):
          return self.V_tilde_s_closed(b=z, b_condition=self.b_star_c)/(self.bh-self.bl)
      result_1 = (self.p_R(b=self.b_star_c, s=self.s)-self.s)*(self.bh- self.b_star_c)/(self.bh-self.bl)
      result_2 = integrate.quad(func=V_s_integrand_closed, a=self.b_dag_c, b=self.b_star_c)[0]
      return result_1 + result_2

  def V_B(self, b, b_star, b_dag):
      if b>=b_star:
          return b-self.p_R(b_star, self.s)
      elif (b<b_star) and (b>=b_dag):
          return self.v_tilde_b(b=b, b_condition=b_star)
      else:
          return 0

  def V_B_closed(self):
      if self.b >= self.b_star_c:
          return self.b-self.p_R(self.b_star_c, self.s)
      elif (self.b < self.b_star_c) and (self.b >= self.b_dag_closed(self.b_star_c)):
          numer = (self.r*(self.b-self.s)+self.c_b+self.c_s)**2
          denom = 2*self.r*(self.r*(self.b_star_c-self.s)+self.c_b+self.c_s)
          return (numer/denom) - (self.c_b/self.r)
      else:
          return 0

  def X(self, b):
      return self.r*(b-self.s)+(2*self.c)

  def Pi_prime(self, b):
      result = self.X(b) - (3*self.X(self.bh)/4)
      result -= ((2*self.c)**1.5)/(4*np.sqrt(self.X(b)))
      result = -result * (2/(3*self.r))

      return (result)*2*self.r/3

  def b_star_closed(self):
      if self.Pi_prime(max(self.bl, self.s)) <=0 :
          return max(self.bl, self.s)
      else:
          opt = root_scalar(self.Pi_prime, bracket=[max(self.bl, self.s), self.bh])
          result = opt.root
          return result

  def b_dag_closed_2(self):
      b_star_c = self.b_star_closed()
      result = self.s + np.sqrt(2*self.kappa*(b_star_c-self.s)+(4*(self.kappa**2)))
      result -=  2*self.kappa
      return result


  def initial_offer_response_buyer(self, b, b_dag, b_star):
      if b>= b_star:
          return 'Accept'
      elif b< b_dag:
          return 'Reject'
      else:
          return f'Delay {self.delta(b=b, b_condition=b_star)} and offer {self.p_R(b, self.s)}'

  def initial_offer_response_buyer_closed(self, b, b_dag, b_star):
      if b>= b_star:
          return 'Accept Immediately'
      elif b< b_dag:
          return 'Reject Immediately'
      else:
          delay = self.delta_c(b=b, b_condition=b_star)
          offer= self.p_R(b, self.s)
          return (delay, offer)



###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################

  def p_R_partial_s(self, b, s):
      if self.r*(b-s)>=np.abs(self.c_s-self.c_b):
          return 1/2
      elif self.r*(b-s)<(self.c_b-self.c_s):
          return 0
      else:
          return 1

  def phi(self, s, s_condition):
      def integrand_phi(z):
          numer = self.p_R_partial_b(b=self.b, s=z)
          denom = self.r*(self.p_R(b=self.b, s=z)-z) + self.c_s
          return numer/denom
      if (s>= s_condition) and (s<= min(self.sh, self.b)):
          result = integrate.quad(func=integrand_phi, a=s_condition, b=s)
          return result[0]
      else:
          return 0

  def exp_r_phi(self, s, s_condition):
      return np.exp(-self.r * self.phi(s=s, s_condition=s_condition))

  def exp_r_phi_c(self, s, s_condition):
      if s >= s_condition:
          numer= self.r*(self.b-s)+(2*self.c)
          denom= self.r*(self.b-s_condition)+(2*self.c)
          return numer/denom
      else:
          return 1

  def phi_c(self, s, s_condition):
      return (np.log(self.exp_r_phi_c(s,s_condition)))/(-self.r)


  def w_tilde_s(self, s, s_condition):
      if (s>=s_condition) and (s<= min(self.sh, self.b)):
          result = self.exp_r_phi(s=s, s_condition=s_condition)*(self.p_R(self.b, s)-s+(self.c_s/self.r))-(self.c_s/self.r)
          return result
      elif (s< s_condition) and (s>= self.sl):
          return self.p_R(self.b, s_condition)-s
      else:
          return 0

  def w_tilde_s_closed(self, s, s_condition):
      if s <= min(self.sh, self.b):
          numer = (self.r*(self.b-s)+self.c_b+self.c_s)**2
          denom = 2*self.r*(self.r*(self.b-s_condition)+self.c_b+self.c_s)
          return (numer/denom) - (self.c_s/self.r)
      else:
          return 0

  def s_dag_opt(self, s_condition):
      if self.w_tilde_s(s=min(self.sh, self.b), s_condition=s_condition) >0:
          return min(self.sh, self.b)
      else:
          def s_dag_constraints(z):
              if self.w_tilde_s(s=z, s_condition=s_condition)<=0:
                  return z
              else:
                  return 1000
          result = minimize_scalar(s_dag_constraints, bounds = (self.sl, min(self.sh, self.b)), method='bounded')
          return result.x


  def s_dag_closed(self, s_condition):
      def E(beta, s_condition):
          result = 2*(self.c_s+(self.r*beta))*(self.r*(self.b-s_condition)+self.c_s+self.c_b)
          return result
      def s_double_dag(beta, s_condition):
          if (beta>=0) and (beta<self.w_tilde_s_closed(s=s_condition, s_condition=s_condition)):
              numer = np.sqrt(E(beta=beta, s_condition=s_condition)) - self.c_s-self.c_b
              return self.b - (numer/self.r)
          elif (beta>=self.w_tilde_s_closed(s=s_condition, s_condition=s_condition)) and (beta<self.w_tilde_s_closed(s=self.sl, s_condition=s_condition)):
              return (s_condition+self.b)/2 - beta
      return s_double_dag(beta=0, s_condition=s_condition)


  def W_tilde_b(self, s, s_condition):
      if s<= s_condition:
          return self.b- self.p_R(self.b, s_condition)

      elif (s<= self.s_dag_opt(s_condition=s_condition)) and (s>s_condition):
          exp = self.exp_r_phi(s=s, s_condition=s_condition)
          result = exp*(self.b-self.p_R(b=self.b, s=s) +(self.c_b/self.r))-(self.c_b/self.r)
          return result
      else:
          return 0

  def W_tilde_b_closed(self, s, s_condition):
      numer = (self.r*(self.b-s)+self.c_b+self.c_s)**2
      denom = 2*self.r*(self.r*(self.b-s_condition)+self.c_b+self.c_s)
      return max((numer/denom)-(self.c/self.r), 0)

  def W_tilde_s_closed(self, s, s_condition):
      return self.W_tilde_b_closed(s,s_condition)

  def W_tilde_s(self, s, s_condition):
      return max(0, self.w_tilde_s(s=s, s_condition=s_condition))

  def W_b_integrand(self, z, s_condition):
      return self.W_tilde_b(s=z, s_condition=s_condition)/(self.sh-self.sl)

  def W_b_obj(self, s_condition):
      result_1 = (self.b-self.p_R(b=self.b, s=s_condition))*(s_condition- self.sl)/(self.sh-self.sl)
      s_dag = self.s_dag_opt(s_condition=s_condition)
      result_2 = integrate.quad(func=self.W_b_integrand, a=s_condition, b=s_dag, args=(s_condition))[0]
      return -result_1 - result_2

  def s_star_W_b(self):
      result = minimize_scalar(fun=self.W_b_obj, bounds=(self.sl, min(self.sh, self.b)), method='bounded')
      if result.success == True:
          return (result.x, -result.fun)
      else:
          raise Exception("b_star optimization failed")


  def Y(self, s):
      return self.r*(self.b-s)+(2*self.c)

  def Pi_prime_S(self, s):
      result = self.Y(s) - (3*self.Y(self.sl)/4)
      result -= ((2*self.c)**1.5)/(4*np.sqrt(self.Y(s)))
      result = result * (2/(3*self.r))

      return result

  def s_star_closed(self):

      if self.Pi_prime_S(min(self.sh, self.b)) >=0 :
          return min(self.sh, self.b)
      else:
          opt = root_scalar(self.Pi_prime_S, bracket=[self.sl, min(self.sh, self.b)])
          result = opt.root
          return result


  def W_s(self, s, s_star, s_dag):
      if (s>=self.sl) and (s<=s_star):
          return self.p_R(b, s_star)-s
      elif (s>s_star) and (s<=s_dag):
          return self.w_tilde_s(s=s, s_condition=s_star)
      else:
          return 0

  def W_s_closed(self, s, s_star, s_dag):
      if (s>=self.sl) and (s<=s_star):
          return self.p_R(self.b, s_star)-s
      elif (s>s_star) and (s<=s_dag):
          numer = (self.r*(self.b-s)+self.c_b+self.c_s)**2
          denom = 2*self.r*(self.r*(self.b-s_star)+self.c_b+self.c_s)
          return (numer/denom)-(self.c/self.r)
      else:
          return 0

  def W_b_closed(self, s, s_star):
      result = (s_star-self.sl)*(self.b-s_star)
      numer = self.Y(s=s_star)**2 - (6*self.c*self.Y(s=s_star)) + (4*self.c*np.sqrt(2*self.c*self.Y(s_star)))
      denom = 3*(self.r**2)
      result += (numer)/denom
      return result / (2*(self.sh-self.sl))


  def initial_offer_response_seller(self, s, s_dag, s_star):
      if s<=s_star:
          return 'Accept'
      elif s>s_dag:
          return 'Reject'
      else:
          return f'Delay {self.phi(s=s, s_condition=s_star)} and offer {self.p_R(self.b, self.s)}'

  def initial_offer_response_seller_closed(self, s, s_dag, s_star):
      if s<=s_star:
          return 'Accept Immediately'
      elif s>s_dag:
          return 'Reject Immediately'
      else:
          delay = self.phi_c(s=s, s_condition=s_star)
          offer= self.p_R(self.b, self.s)
          return (delay, offer)


###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################
###########################################################

  def run_print_buyer(self, method):
      delay_c = None
      print('')
      print('')
      print('###########################################')
      print('Initial Paramters')
      print(f'b = {self.b}, s = {self.s}, c_s = {self.c_s}, c_b = {self.c_b}, r = {self.r}')
      if method in ['opt', 'both']:
          o_start = time.time()
          (b_star_o, V_s_o) = self.b_star_V_s()
          b_dag_o = self.b_dag(b_condition=b_star_o)
          initial_price_o = self.p_R(b=b_dag_o, s=self.s)
          V_b_o = self.V_B(b=self.b, b_star=b_star_o, b_dag=b_dag_o)
          response_o = self.initial_offer_response_buyer(self.b, b_dag_o, b_star_o)

          o_end = time.time()
          print('---------------------------------')
          print(f'Results from optimization: ({o_end-o_start} seconds taken)')
          print(f'b_dag = {b_dag_o}')
          print(f'b_star = {b_star_o}')
          print(f'Initial price = {initial_price_o}')
          print(f'Initial response = {response_o}')
          if (self.b >= b_dag_o) and (self.b<b_star_o):
              delay_o = self.delta(b=self.b, b_condition=b_star_o)
              print(f'Delta: {delay_o}')
          print(f'V_B = {V_b_o}')
          print(f'V_S = {V_s_o}')
          print('')


      if method in ['closed', 'both']:
          c_start = time.time()
          b_star_c = self.b_star_closed()
          b_dag_c = self.b_dag_closed(b_condition=b_star_c)
          initial_price_c = self.p_R(b=b_dag_c, s=self.s)
          V_b_c = self.V_B_closed()
          V_s_c = self.V_S_closed()
          c_end = time.time()
          response_c = self.initial_offer_response_buyer_closed(self.b, b_dag_c, b_star_c)


          print('----------------------------------')
          print(f'Results from closed form:  {c_end-c_start} seconds taken')
          print(f'b_dag = {b_dag_c}')
          print(f'b_star = {b_star_c}')
          print(f'Initial price = {initial_price_c}')
          print(f'Initial response = {response_c}')
          if (self.b >= b_dag_c) and (self.b<b_star_c):
              delay_c = self.delta_c(b=self.b, b_condition=b_star_c)
              print(f'Delta: {delay_c}')
          print(f'V_B = {V_b_c}')
          print(f'V_S = {V_s_c}')

          print('----------------------------------')
          print('Difference')
          print(f'b_dag: {b_dag_c-b_dag_o}')
          print(f'b_star: {b_star_c-b_star_o}')
          if delay_c != None:
              print(f'Delta: {delay_c - delay_o}')
          print(f'V_B: {V_b_c-V_b_o}')
          print(f'V_S: {V_s_c-V_s_o}')

  def run_print_seller(self, method):
      delay_c = None
      print('')
      print('')
      print('###########################################')
      print('Initial Paramters')
      print(f'b = {self.b}, s = {self.s}, c_s = {self.c_s}, c_b = {self.c_b}, r = {self.r}')
      if method in ['opt', 'both']:
          o_start = time.time()
          (s_star_o, W_b_o) = self.s_star_W_b()
          s_dag_o = self.s_dag_opt(s_condition=s_star_o)
          initial_price_o = self.p_R(b=self.b, s=s_star_o)
          W_s_o = self.W_s(s=self.s, s_star=s_star_o, s_dag=s_dag_o)
          response_o = self.initial_offer_response_seller(s=self.s, s_dag=s_dag_o, s_star=s_star_o)

          o_end = time.time()
          print('---------------------------------')
          print(f'Results from optimization: ({o_end-o_start} seconds taken)')
          print(f's_dag = {s_dag_o}')
          print(f's_star = {s_star_o}')
          print(f'Initial price = {initial_price_o}')
          print(f'Initial response = {response_o}')
          if (self.s <= s_dag_o) and (self.s>s_star_o):
              delay_o = self.phi(s=self.s, s_condition=s_star_o)
              print(f'Delta: {delay_o}')
          print(f'W_S = {W_s_o}')
          print(f'W_B = {W_b_o}')
          print('')


      if method in ['closed', 'both']:
          o_start = time.time()
          s_star_c = self.s_star_closed()
          s_dag_c = self.s_dag_closed(s_condition=s_star_c)
          initial_price_c = self.p_R(b=self.b, s=s_star_c)
          W_s_c = self.W_s_closed(s=self.s, s_star=s_star_c, s_dag=s_dag_c)
          W_b_c = self.W_b_closed(s=self.s, s_star=s_star_c)

          response_c = self.initial_offer_response_seller_closed(s=self.s, s_dag=s_dag_o, s_star=s_star_o)

          o_end = time.time()
          print('---------------------------------')
          print(f'Results from closed: ({o_end-o_start} seconds taken)')
          print(f's_dag = {s_dag_c}')
          print(f's_star = {s_star_c}')
          print(f'Initial price = {initial_price_c}')
          print(f'Initial response = {response_c}')
          if (self.s <= s_dag_c) and (self.s>s_star_c):
              delay_c = self.phi_c(s=self.s, s_condition=s_star_c)
              print(f'Delta: {delay_o}')
          print(f'W_S = {W_s_c}')
          print(f'W_B = {W_b_c}')
          print('')

          print('----------------------------------')
          print('Difference')
          print(f's_dag: {s_dag_c-s_dag_o}')
          print(f's_star: {s_star_c-s_star_o}')
          if delay_c != None:
              print(f'Delta: {delay_c - delay_o}')
          print(f'W_B: {W_b_c-W_b_o}')
          print(f'W_S: {W_s_c-W_s_o}')
      breakpoint()

  def run(self, method):
      result = self.__dict__
      result['closed'] = {}
      result['opt'] = {}

      if method in ['closed', 'both']:
          b_dag_c = self.b_dag_c
          b_star_c = self.b_star_closed()
          initial_price_buyer = self.p_R(b=b_star_c, s=self.s)
          initial_offer_response_buyer = self.initial_offer_response_buyer_closed(b_star=b_star_c, b_dag=b_dag_c, b=self.b)
          V_B_c = self.V_B_closed()
          V_S_c = self.V_S_closed()
          if self.b > b_star_c:
              exp_r_delta = 1
              delta = 0
          elif self.b < b_dag_c:
              exp_r_delta = 0
              delta = self.delta_c(b=self.b_dag_c, b_condition=self.b_star_c)
          else:
              exp_r_delta = self.exp_r_delta_c(b=self.b, b_condition=b_star_c)
              delta = self.delta_c(b=self.b, b_condition=self.b_star_c)

          expected_exp_r_delta = self.expected_exp_r_delta()

          result['closed']['buyer'] = {}
          result['closed']['buyer']['b_dag'] = b_dag_c
          result['closed']['buyer']['b_star'] = b_star_c
          result['closed']['buyer']['initial_offer'] = initial_price_buyer
          result['closed']['buyer']['initial_offer_response'] =initial_offer_response_buyer
          result['closed']['buyer']['exp_r_delta'] = exp_r_delta
          result['closed']['buyer']['expected_exp_r_delta'] = expected_exp_r_delta
          result['closed']['buyer']['delta'] = delta
          result['closed']['buyer']['V_B'] = V_B_c
          result['closed']['buyer']['V_S'] = V_S_c

      if method in ['opt', 'both']:
          (b_star_o, V_S_o) = self.b_star_V_s()
          b_dag_o = self.b_dag(b_condition=b_star_o)
          initial_price_buyer_o = self.p_R(b=b_dag_o, s=self.s)
          V_B_o = self.V_B(b=self.b, b_star=b_star_o, b_dag=b_dag_o)
          initial_offer_response_buyer_o = self.initial_offer_response_buyer(self.b, b_dag_o, b_star_o)
          if self.b > b_star_o:
              exp_r_delta_o = 1
              delta_o = 0
          elif self.b < b_dag_o:
              exp_r_delta_o = 0
              delta_o = self.delta(b=b_dag_o, b_condition=b_star_o)
          else:
              exp_r_delta_o = self.exp_r_delta(b=self.b, b_condition=b_star_o)
              delta_o = np.log(exp_r_delta_o)/(-self.r)

          expected_exp_r_delta_o = self.expected_exp_r_delta()


          result['opt']['buyer'] = {}
          result['opt']['buyer']['b_dag'] = b_dag_o
          result['opt']['buyer']['b_star'] = b_star_o
          result['opt']['buyer']['initial_offer'] = initial_price_buyer_o
          result['opt']['buyer']['initial_offer_response'] =initial_offer_response_buyer_o
          result['opt']['buyer']['exp_r_delta'] = exp_r_delta_o
          result['opt']['buyer']['expected_exp_r_delta'] = expected_exp_r_delta_o
          result['opt']['buyer']['delta'] = delta_o
          result['opt']['buyer']['V_B'] = V_B_o
          result['opt']['buyer']['V_S'] = V_S_o

      return result


In [7]:
args = argparse.Namespace()

B=np.array(0.8)
S=np.array(0)
R=np.array(0.01)
C = np.array(0.05)
MIN_B=np.array(0)
MAX_B=np.array(30)
MIN_S=np.array(0)
MAX_S=np.array(30)
BL=np.array(0)
BH=np.array(30)
SL=np.array(0)
SH=np.array(30)


args.b = B
args.s = S
args.r = R
args.c = C
args.MIN_B = MIN_B
args.MAX_B = MAX_B
args.MIN_S = MIN_S
args.MAX_S = MAX_S
args.sl = SL
args.sh = SH
args.bl = BL
args.bh = BH

In [25]:
buyer_valuation = []
deltas = []
for b in range(0, 30, 1):
    args.b = b
    onesided = OneSided(args)
    hi = onesided.run(method='opt')
    buyer_valuation.append(hi["opt"]["buyer"]["b_star"])
    deltas.append(hi["opt"]["buyer"]["delta"])


In [26]:
deltas

[57.22800669254552,
 57.22800669254552,
 57.22800669254552,
 57.22800669254552,
 57.22800669254552,
 57.22800669254552,
 57.22800669254552,
 57.22800669254552,
 np.float64(55.67734689487912),
 np.float64(50.27062476785154),
 np.float64(45.141295329096494),
 np.float64(40.262278912153306),
 np.float64(35.610277348664006),
 np.float64(31.16510109158063),
 np.float64(26.909139649701036),
 np.float64(22.82694019767553),
 np.float64(18.904868882347394),
 np.float64(15.130836084062684),
 np.float64(11.494071666975207),
 np.float64(7.984939685848191),
 np.float64(4.594784518280062),
 np.float64(1.3158022359809702),
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [24]:
hi["opt"]["buyer"]["delta"]

57.22800669254552